In [2]:
#import the necessary libraries for this project
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

In [26]:
#import the data
from pathlib import Path
import os
project_root = Path.cwd()
if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent
print("Project root:", project_root)
df = pd.read_csv(project_root / "data" / "raw" / "IMDB Dataset.csv")

#check the missing data and the data types
print(df.isnull().sum())
print(df.dtypes)

print("inspecting the shape")
print(df.shape)

df.head()

#check the sentiment labels
print(df['sentiment'].value_counts())

c:\Users\Mumba Mumba\Desktop\tensor flow\projects\01_binary_sentiment_analysis\data\notebook
review       0
sentiment    0
dtype: int64
review       str
sentiment    str
dtype: object
inspecting the shape
(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
#convert the text to numbers

from sklearn.feature_extraction.text import  CountVectorizer

vectorizer = CountVectorizer(max_features=20_000)

X = vectorizer.fit_transform(df['review'].fillna(""))
print(X.shape)

#convert sentiment labels into numbers
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])

print("Label mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
print("First 10 labels:", y[:10])

(50000, 101895)
Label mapping: {'negative': np.int64(0), 'positive': np.int64(1)}
First 10 labels: [1 1 1 0 1 1 1 0 0 1]


In [36]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training reviews:", X_train.shape)
print("Testing reviews:", X_test.shape)


Training reviews: (40000, 101895)
Testing reviews: (10000, 101895)


In [ ]:
#convert the data into tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print(y_train_tensor.shape)
print(y_train_tensor.dtype)



torch.Size([40000])
torch.float32
torch.Size([4, 101895])
torch.float32


In [41]:

# Select the first 32 training reviews
X_batch = X_train[:32].toarray()

# Convert the reviews into a PyTorch tensor
X_batch_tensor = torch.tensor(X_batch, dtype=torch.float32)

# Select the matching sentiment labels
y_batch_tensor = torch.tensor(y_train[:32], dtype=torch.float32)

print("Input shape:", X_batch_tensor.shape)
print("Labels shape:", y_batch_tensor.shape)

Input shape: torch.Size([32, 101895])
Labels shape: torch.Size([32])


In [48]:

from torch.utils.data import Dataset, DataLoader

# Convert sentiment labels into PyTorch tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create a Dataset that retrieves one review and its label
class ReviewDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        review = torch.tensor(
            self.features[index].toarray().squeeze(),
            dtype=torch.float32
        )
        label = self.labels[index]
        return review, label

# Create training and testing datasets
train_dataset = ReviewDataset(X_train, y_train_tensor)
test_dataset = ReviewDataset(X_test, y_test_tensor)

# Group reviews into batches of 32
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# Inspect the first training batch
X_batch, y_batch = next(iter(train_loader))

print("Review batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)


Review batch shape: torch.Size([32, 101895])
Label batch shape: torch.Size([32])


In [50]:
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

print(model)

Sequential(
  (0): Linear(in_features=101895, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=1, bias=True)
)


In [54]:
loss = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [56]:
epochs = 1

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = model(X_batch).squeeze(1)
        batch_loss = loss(predictions, y_batch)

        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Loss: {average_loss:.4f}"
    )

Epoch 1/1 | Loss: 0.0701


In [57]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch).squeeze(1)
        predicted_labels = (predictions >= 0).float()

        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total

print(f"Test accuracy: {accuracy:.2%}")

Test accuracy: 88.90%


In [71]:
model.eval()

review = input("Enter a movie review: ")

review_vector = vectorizer.transform([review])
review_tensor = torch.tensor(
    review_vector.toarray(),
    dtype=torch.float32
)

with torch.no_grad():
    logit = model(review_tensor)
    probability = torch.sigmoid(logit).item()

sentiment = "Positive" if probability >= 0.5 else "Negative"

print(f"Sentiment: {sentiment}")
print(f"Positive probability: {probability:.2%}")

Sentiment: Positive
Positive probability: 56.96%


In [72]:
import os
import joblib
import torch

artifact_dir = project_root / "models" / "notebook_experiment"
artifact_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), artifact_dir / "sentiment_model.pth")
joblib.dump(vectorizer, artifact_dir / "vectorizer.joblib")

print(f"Notebook artifacts saved to {artifact_dir}")

Model and vectorizer saved successfully.
